In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "gold") 

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalogo}.{esquema_sink}.gold_sales_monthly AS
WITH monthly AS (
    SELECT
        date_trunc('month', order_purchase_timestamp) AS order_month,
        product_category_name_english               AS category,
        COUNT(DISTINCT order_id)                     AS num_orders,
        SUM(price + freight_value)                   AS revenue,
        SUM(price)                                   AS revenue_ex_freight
    FROM {catalogo}.{esquema_source}.orders_transformed
    WHERE order_status NOT IN ('canceled', 'unavailable')
    GROUP BY 1, 2
)
SELECT
    *,
    ROUND(revenue / num_orders, 2) AS avg_order_value,
    ROUND(
        (revenue - LAG(revenue) OVER (PARTITION BY category ORDER BY order_month))
        / LAG(revenue) OVER (PARTITION BY category ORDER BY order_month) * 100
    , 1) AS mom_growth_pct
FROM monthly
ORDER BY category, order_month
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalogo}.{esquema_sink}.gold_delivery_performance AS
WITH order_category AS (
    SELECT DISTINCT
        order_id,
        customer_state,
        product_category_name_english AS category,
        order_estimated_delivery_date,
        order_delivered_customer_date
    FROM {catalogo}.{esquema_source}.orders_transformed
    WHERE order_delivered_customer_date IS NOT NULL
),
delivery_calc AS (
    SELECT
        *,
        datediff(order_delivered_customer_date, order_estimated_delivery_date) AS delay_days,
        CASE WHEN order_delivered_customer_date > order_estimated_delivery_date
             THEN 1 ELSE 0 END AS is_late
    FROM order_category
)
SELECT
    category,
    customer_state,
    COUNT(DISTINCT order_id)                                   AS num_orders,
    ROUND(AVG(is_late) * 100, 1)                                AS pct_late,
    ROUND(AVG(delay_days), 1)                                   AS avg_delay_days,
    ROUND(AVG(CASE WHEN is_late = 1 THEN delay_days END), 1)    AS avg_delay_days_when_late
FROM delivery_calc
GROUP BY 1, 2
ORDER BY pct_late DESC
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalogo}.{esquema_sink}.gold_geo_concentration AS
WITH by_state AS (
    SELECT
        customer_state,
        COUNT(DISTINCT order_id)             AS num_orders,
        COUNT(DISTINCT customer_unique_id)   AS num_customers,
        SUM(price + freight_value)           AS revenue
    FROM {catalogo}.{esquema_source}.orders_transformed
    WHERE order_status NOT IN ('canceled', 'unavailable')
    GROUP BY customer_state
)
SELECT
    'Brazil' AS country,
    customer_state,
    num_orders,
    num_customers,
    revenue,
    ROUND(revenue / SUM(revenue) OVER () * 100, 2) AS pct_of_total_revenue,
    ROUND(SUM(POWER(revenue / SUM(revenue) OVER () * 100, 2)) OVER (), 1) AS hhi_national
FROM by_state
ORDER BY revenue DESC
""")